In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    when,
    lit,
    to_date,
    date_trunc,
    hour,
    month,
    dayofweek,
    sum
)

StatementMeta(sparkpoolenergy, 16, 2, Finished, Available, Finished, False)

In [3]:
spark = SparkSession.builder \
    .appName("BuildEnergyBaseDataset") \
    .getOrCreate()

StatementMeta(sparkpoolenergy, 16, 3, Finished, Available, Finished, False)

In [4]:
#B. LECTURE DES DONNÉES

StatementMeta(sparkpoolenergy, 16, 4, Finished, Available, Finished, False)

In [5]:
#Lire dataset énergie principale 
energy_df = spark.read.parquet(
    "abfss://processed@energybigdatastorage.dfs.core.windows.net/halfhourly_v2/"
)

StatementMeta(sparkpoolenergy, 16, 5, Finished, Available, Finished, False)

In [6]:
#Lire météo
weather_df = spark.read.parquet(
    "abfss://processed@energybigdatastorage.dfs.core.windows.net/weather_hourly/"
)

StatementMeta(sparkpoolenergy, 16, 6, Finished, Available, Finished, False)

In [7]:
#Lire households
households_df = spark.read.parquet(
    "abfss://processed@energybigdatastorage.dfs.core.windows.net/informations_households/"
)

StatementMeta(sparkpoolenergy, 16, 7, Finished, Available, Finished, False)

In [8]:
#Lire holidays
holidays_df = spark.read.parquet(
    "abfss://processed@energybigdatastorage.dfs.core.windows.net/uk_bank_holidays/"
)

StatementMeta(sparkpoolenergy, 16, 8, Finished, Available, Finished, False)

In [9]:
#C. EXPLORATION

StatementMeta(sparkpoolenergy, 16, 9, Finished, Available, Finished, False)

In [10]:
#Vérifier chargement
holidays_df.show(5)

StatementMeta(sparkpoolenergy, 16, 10, Finished, Available, Finished, False)

+-------------+-------------------+----+
|Bank_holidays|               Type|year|
+-------------+-------------------+----+
|   2014-04-18|        Good Friday|2014|
|   2014-01-01|     New Year's Day|2014|
|   2013-01-04|      Easter Monday|2013|
|   2012-08-27|Summer bank holiday|2012|
|   2013-12-26|         Boxing Day|2013|
+-------------+-------------------+----+
only showing top 5 rows



In [11]:
energy_df.printSchema()

StatementMeta(sparkpoolenergy, 16, 11, Finished, Available, Finished, False)

root
 |-- LCLid: string (nullable = true)
 |-- tstp: timestamp (nullable = true)
 |-- energy_kwh: float (nullable = true)
 |-- is_outlier: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)



In [12]:
#Vérifier nombre de lignes
energy_df.count()

StatementMeta(sparkpoolenergy, 16, 12, Finished, Available, Finished, False)

167817021

In [13]:
#Vérifier valeurs nulles

energy_df.select([
    col(c).isNull().alias(c)
    for c in energy_df.columns
]).show()

StatementMeta(sparkpoolenergy, 16, 13, Finished, Available, Finished, False)

+-----+-----+----------+----------+-----+-----------+----------+-------+-----+-----+
|LCLid| tstp|energy_kwh|is_outlier| hour|day_of_week|is_weekend|quarter| year|month|
+-----+-----+----------+----------+-----+-----------+----------+-------+-----+-----+
|false|false|     false|     false|false|      false|     false|  false|false|false|
|false|false|     false|     false|false|      false|     false|  false|false|false|
|false|false|     false|     false|false|      false|     false|  false|false|false|
|false|false|     false|     false|false|      false|     false|  false|false|false|
|false|false|     false|     false|false|      false|     false|  false|false|false|
|false|false|     false|     false|false|      false|     false|  false|false|false|
|false|false|     false|     false|false|      false|     false|  false|false|false|
|false|false|     false|     false|false|      false|     false|  false|false|false|
|false|false|     false|     false|false|      false|     false| 

In [14]:
#Vérifier timestamps
energy_df.select("tstp").show(5, False)

StatementMeta(sparkpoolenergy, 16, 14, Finished, Available, Finished, False)

+-------------------+
|tstp               |
+-------------------+
|2012-12-01 00:00:00|
|2012-12-01 00:30:00|
|2012-12-01 01:00:00|
|2012-12-01 01:30:00|
|2012-12-01 02:00:00|
+-------------------+
only showing top 5 rows



In [15]:
#E. ALIGNEMENT TEMPOREL

StatementMeta(sparkpoolenergy, 16, 15, Finished, Available, Finished, False)

In [16]:
#Créer weather_hour
energy_df = energy_df.withColumn(
    "weather_hour",
    date_trunc("hour", col("tstp"))
)

StatementMeta(sparkpoolenergy, 16, 16, Finished, Available, Finished, False)

In [17]:
energy_df.select(
    "tstp",
    "weather_hour"
).show(10, False)

StatementMeta(sparkpoolenergy, 16, 17, Finished, Available, Finished, False)

+-------------------+-------------------+
|tstp               |weather_hour       |
+-------------------+-------------------+
|2012-12-01 00:00:00|2012-12-01 00:00:00|
|2012-12-01 00:30:00|2012-12-01 00:00:00|
|2012-12-01 01:00:00|2012-12-01 01:00:00|
|2012-12-01 01:30:00|2012-12-01 01:00:00|
|2012-12-01 02:00:00|2012-12-01 02:00:00|
|2012-12-01 02:30:00|2012-12-01 02:00:00|
|2012-12-01 03:00:00|2012-12-01 03:00:00|
|2012-12-01 03:30:00|2012-12-01 03:00:00|
|2012-12-01 04:00:00|2012-12-01 04:00:00|
|2012-12-01 04:30:00|2012-12-01 04:00:00|
+-------------------+-------------------+
only showing top 10 rows



In [18]:
#F. JOINTURES

StatementMeta(sparkpoolenergy, 16, 18, Finished, Available, Finished, False)

In [19]:
#1. Jointure météo
base_df = energy_df.join(
    weather_df,
    energy_df.weather_hour == weather_df.timestamp,
    "left"
)

StatementMeta(sparkpoolenergy, 16, 19, Finished, Available, Finished, False)

In [20]:
#2. Jointure households
base_df = base_df.join(
    households_df,
    on="LCLid",
    how="left"
)

StatementMeta(sparkpoolenergy, 16, 20, Finished, Available, Finished, False)

In [21]:
#Ajouter date
base_df = base_df.withColumn(
    "date",
    to_date("timestamp")
)

StatementMeta(sparkpoolenergy, 16, 21, Finished, Available, Finished, False)

In [22]:
#4. Jointure holidays

holidays_df = holidays_df.withColumn(
    "date",
    to_date("Bank_holidays")
)


StatementMeta(sparkpoolenergy, 16, 22, Finished, Available, Finished, False)

In [23]:
base_df = base_df.join(
    holidays_df,
    on="date",
    how="left"
)

StatementMeta(sparkpoolenergy, 16, 23, Finished, Available, Finished, False)

In [24]:
base_df.count()

StatementMeta(sparkpoolenergy, 16, 24, Finished, Available, Finished, False)

343366876

In [25]:
#G. FEATURES SIMPLES : Créer les premières colonnes utiles au ML.

StatementMeta(sparkpoolenergy, 16, 25, Finished, Available, Finished, False)

In [26]:
#1. is_holiday
base_df = base_df.withColumn(
    "is_holiday",
    when(
        col("Bank_holidays").isNotNull(),
        1
    ).otherwise(0)
)

StatementMeta(sparkpoolenergy, 16, 26, Finished, Available, Finished, False)

In [27]:
#2. Acorn_grouped

base_df = base_df.withColumn(
    "Acorn_grouped",
    when(col("Acorn").isin("A","B","C"), "High")
    .when(col("Acorn").isin("D","E"), "Medium")
    .otherwise("Low")
)

StatementMeta(sparkpoolenergy, 16, 27, Finished, Available, Finished, False)

In [28]:
#H. VALIDATION
base_df.printSchema()

StatementMeta(sparkpoolenergy, 16, 28, Finished, Available, Finished, False)

root
 |-- date: date (nullable = true)
 |-- LCLid: string (nullable = true)
 |-- tstp: timestamp (nullable = true)
 |-- energy_kwh: float (nullable = true)
 |-- is_outlier: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- weather_hour: timestamp (nullable = true)
 |-- visibility: double (nullable = true)
 |-- windBearing: double (nullable = true)
 |-- temperature: double (nullable = true)
 |-- time: timestamp (nullable = true)
 |-- dewPoint: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- apparentTemperature: double (nullable = true)
 |-- windSpeed: double (nullable = true)
 |-- precipType: string (nullable = true)
 |-- icon: string (nullable = true)
 |-- humidity: double (nullable = true)
 |-- summary: string (nullable = true)
 |-- timestamp: tim

In [29]:
base_df.show(5)

StatementMeta(sparkpoolenergy, 16, 29, Finished, Available, Finished, False)

+----------+---------+-------------------+----------+----------+----+-----------+----------+-------+----+-----+-------------------+----------+-----------+-----------+-------------------+--------+--------+-------------------+---------+----------+-----------+--------+-------+-------------------+----+---+---------+----------+-------------------+---------------------------+----------------+-----------------+-------------------+------------------+----------------+--------------+----+-----+--------+-------+-------------+--------+-------------+----+----+----------+
|      date|    LCLid|               tstp|energy_kwh|is_outlier|hour|day_of_week|is_weekend|quarter|year|month|       weather_hour|visibility|windBearing|temperature|               time|dewPoint|pressure|apparentTemperature|windSpeed|precipType|       icon|humidity|summary|          timestamp|hour|day|dayofweek|is_weekend|temperature_imputed|apparentTemperature_imputed|humidity_imputed|windSpeed_imputed|windBearing_imputed|visibili

In [30]:
base_df.select(
    "tstp",
    "energy_kwh",
    "LCLid"
).filter(
    col("tstp").isNull() |
    col("energy_kwh").isNull() |
    col("LCLid").isNull()
).show(10, False)

StatementMeta(sparkpoolenergy, 16, 30, Finished, Available, Finished, False)

+-------------------+----------+---------+
|tstp               |energy_kwh|LCLid    |
+-------------------+----------+---------+
|2012-12-19 12:32:39|NULL      |MAC005559|
|2012-12-19 12:32:39|NULL      |MAC005559|
|2012-12-19 12:32:39|NULL      |MAC005556|
|2012-12-19 12:32:39|NULL      |MAC005556|
|2012-12-18 15:18:21|NULL      |MAC001150|
|2012-12-18 15:18:21|NULL      |MAC001150|
|2012-12-19 12:32:41|NULL      |MAC005563|
|2012-12-19 12:32:41|NULL      |MAC005563|
|2012-12-19 12:32:40|NULL      |MAC005560|
|2012-12-19 12:32:40|NULL      |MAC005560|
+-------------------+----------+---------+
only showing top 10 rows



In [31]:
base_df = base_df.dropna(subset=["energy_kwh"])

StatementMeta(sparkpoolenergy, 16, 31, Finished, Available, Finished, False)

In [32]:
from pyspark.sql.functions import col, sum

base_df.select(
    sum(col("tstp").isNull().cast("int")).alias("null_tstp"),
    sum(col("energy_kwh").isNull().cast("int")).alias("null_energy"),
    sum(col("LCLid").isNull().cast("int")).alias("null_household")
).show()

StatementMeta(sparkpoolenergy, 16, 32, Finished, Available, Finished, False)

+---------+-----------+--------------+
|null_tstp|null_energy|null_household|
+---------+-----------+--------------+
|        0|          0|             0|
+---------+-----------+--------------+



In [33]:
#4. Vérifier duplication après jointure
base_df.groupBy(
    "LCLid",
    "tstp"
).count().filter(
    col("count") > 1
).show()

StatementMeta(sparkpoolenergy, 16, 33, Finished, Available, Finished, False)

+---------+-------------------+-----+
|    LCLid|               tstp|count|
+---------+-------------------+-----+
|MAC000053|2012-12-09 23:30:00|    2|
|MAC000071|2012-12-05 01:30:00|    2|
|MAC000071|2012-12-06 06:30:00|    2|
|MAC000164|2012-12-05 11:00:00|    2|
|MAC000164|2012-12-07 05:00:00|    2|
|MAC000201|2012-12-01 10:00:00|    2|
|MAC000201|2012-12-02 17:00:00|    2|
|MAC000201|2012-12-06 00:00:00|    2|
|MAC000201|2012-12-06 14:30:00|    2|
|MAC000259|2012-12-01 20:00:00|    2|
|MAC000259|2012-12-02 21:30:00|    2|
|MAC000259|2012-12-07 11:00:00|    2|
|MAC000259|2012-12-08 22:00:00|    2|
|MAC000307|2012-12-03 14:00:00|    2|
|MAC000307|2012-12-09 06:30:00|    2|
|MAC000322|2012-12-01 18:30:00|    2|
|MAC000322|2012-12-08 15:00:00|    2|
|MAC000322|2012-12-09 02:30:00|    2|
|MAC000406|2012-12-03 17:30:00|    2|
|MAC000406|2012-12-11 10:30:00|    2|
+---------+-------------------+-----+
only showing top 20 rows



In [34]:
#Supprimer les doublons sur une clé précise
base_df = base_df.dropDuplicates(["LCLid", "tstp"])

StatementMeta(sparkpoolenergy, 16, 34, Finished, Available, Finished, False)

In [35]:
#5. Vérifier volume
initial_count = energy_df.count()
print("Avant jointures :", initial_count)

final_count = base_df.count()
print("Après jointures :", final_count)

print("Différence :", final_count - initial_count)
print("Ratio :", final_count / initial_count)

StatementMeta(sparkpoolenergy, 16, 35, Finished, Available, Finished, False)

Avant jointures : 167817021
Après jointures : 167817009
Différence : -12
Ratio : 0.9999999284935466


In [36]:
checkpoint_df = base_df

StatementMeta(sparkpoolenergy, 16, 36, Finished, Available, Finished, False)

In [37]:
base_df.printSchema()

StatementMeta(sparkpoolenergy, 16, 37, Finished, Available, Finished, False)

root
 |-- date: date (nullable = true)
 |-- LCLid: string (nullable = true)
 |-- tstp: timestamp (nullable = true)
 |-- energy_kwh: float (nullable = true)
 |-- is_outlier: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- weather_hour: timestamp (nullable = true)
 |-- visibility: double (nullable = true)
 |-- windBearing: double (nullable = true)
 |-- temperature: double (nullable = true)
 |-- time: timestamp (nullable = true)
 |-- dewPoint: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- apparentTemperature: double (nullable = true)
 |-- windSpeed: double (nullable = true)
 |-- precipType: string (nullable = true)
 |-- icon: string (nullable = true)
 |-- humidity: double (nullable = true)
 |-- summary: string (nullable = true)
 |-- timestamp: tim

In [38]:
from collections import Counter

dups = [col for col, count in Counter(base_df.columns).items() if count > 1]

print(dups)

StatementMeta(sparkpoolenergy, 16, 38, Finished, Available, Finished, False)

['hour', 'is_weekend', 'year', 'month']


In [39]:
base_df = base_df.drop(
    "hour",
    "is_weekend",
    "year",
    "month"
)

StatementMeta(sparkpoolenergy, 16, 39, Finished, Available, Finished, False)

In [40]:
from pyspark.sql.functions import (
    hour,
    month,
    year,
    dayofweek,
    when
)

base_df = base_df \
    .withColumn("hour", hour("tstp")) \
    .withColumn("month", month("tstp")) \
    .withColumn("year", year("tstp")) \
    .withColumn("day_of_week", dayofweek("tstp")) \
    .withColumn(
        "is_weekend",
        when(dayofweek("tstp").isin([1, 7]), 1).otherwise(0)
    )

StatementMeta(sparkpoolenergy, 16, 40, Finished, Available, Finished, False)

In [41]:
from collections import Counter

dups = [c for c, n in Counter(base_df.columns).items() if n > 1]

print(dups)

StatementMeta(sparkpoolenergy, 16, 41, Finished, Available, Finished, False)

[]


In [42]:
#le dataset propre final
base_df_clean = base_df.select(
    "LCLid",
    "tstp",
    "energy_kwh",
    "is_outlier",

    # time
    "hour",
    "day_of_week",
    "month",
    "year",
    "is_weekend",

    # weather
    "temperature",
    "humidity",
    "windSpeed",
    "pressure",
    "precipType",

    # household
    "stdorToU",
    "Acorn",
    "Acorn_grouped",

    # holidays
    "is_holiday"
)

StatementMeta(sparkpoolenergy, 16, 42, Finished, Available, Finished, False)

In [43]:
base_df_clean.printSchema()

StatementMeta(sparkpoolenergy, 16, 43, Finished, Available, Finished, False)

root
 |-- LCLid: string (nullable = true)
 |-- tstp: timestamp (nullable = true)
 |-- energy_kwh: float (nullable = true)
 |-- is_outlier: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- is_weekend: integer (nullable = false)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- windSpeed: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- precipType: string (nullable = true)
 |-- stdorToU: string (nullable = true)
 |-- Acorn: string (nullable = true)
 |-- Acorn_grouped: string (nullable = false)
 |-- is_holiday: integer (nullable = false)



In [45]:
base_df_clean.write \
    .mode("overwrite") \
    .format("delta") \
    .save(
       "abfss://curated@energybigdatastorage.dfs.core.windows.net/energy_base_dataset"
    )

StatementMeta(sparkpoolenergy, 16, 45, Finished, Available, Finished, False)

In [46]:
#3. Vérifier sauvegarde
test_df = spark.read.format("delta").load(
    "abfss://curated@energybigdatastorage.dfs.core.windows.net/energy_base_dataset"
)

test_df.show(5)

StatementMeta(sparkpoolenergy, 16, 46, Finished, Available, Finished, False)

+---------+-------------------+----------+----------+----+-----------+-----+----+----------+-----------+--------+---------+--------+----------+--------+-------+-------------+----------+
|    LCLid|               tstp|energy_kwh|is_outlier|hour|day_of_week|month|year|is_weekend|temperature|humidity|windSpeed|pressure|precipType|stdorToU|  Acorn|Acorn_grouped|is_holiday|
+---------+-------------------+----------+----------+----+-----------+-----+----+----------+-----------+--------+---------+--------+----------+--------+-------+-------------+----------+
|MAC000002|2012-10-14 11:30:00|     0.667|         1|  11|          1|   10|2012|         1|      11.07|    0.68|     3.12| 1001.99|      rain|     Std|ACORN-A|          Low|         0|
|MAC000002|2012-10-15 23:30:00|     0.237|         0|  23|          2|   10|2012|         0|      11.56|    0.92|     4.44|  998.97|      rain|     Std|ACORN-A|          Low|         0|
|MAC000002|2012-10-21 04:00:00|     0.251|         0|   4|          1|